In [2]:
def factset_icb_mapping(df, path_params = r"C:\GoogleDrive\TP\00_screen\factset_icb_mapping.xlsx"):
    
    # 读取映射文件
    df_mapping = pd.read_excel(path_params, sheet_name='Mapping', header=0, na_values="@NA")
    df_mapping.rename(columns={
        'Benchmark ICB Supersector 19': ' Benchmark ICB Supersector ',
        'Benchmark ICB Industry 11': ' Benchmark ICB Industry '
    }, inplace=True)
    
    # 创建映射字典 - 添加去重处理
    # 1. Supersector -> ICB19_ID
    mapping_sector_to_id = (df_mapping.dropna(subset=['ICB19_ID'])
                            .drop_duplicates(subset=[' Benchmark ICB Supersector '])
                            .set_index(' Benchmark ICB Supersector ')['ICB19_ID']
                            .to_dict())
    
    # 2. FactSet Ind -> ICB19
    mapping_factset_to_icb19 = (df_mapping
                                .drop_duplicates(subset=['FactSet Ind'])
                                .set_index('FactSet Ind')['Transco_ICB_19']
                                .to_dict())
    
    # 3. Industry -> ICB11_ID
    mapping_industry_to_id = (df_mapping.dropna(subset=['ICB11_ID'])
                              .drop_duplicates(subset=[' Benchmark ICB Industry '])
                              .set_index(' Benchmark ICB Industry ')['ICB11_ID']
                              .to_dict())
    
    # 4. ICB19 -> ICB11 (这里最可能出现重复)
    mapping_icb19_to_icb11 = (df_mapping
                              .drop_duplicates(subset=['ICB_19_mapping'])
                              .set_index('ICB_19_mapping')['Transco_ICB_11']
                              .to_dict())
    
    # 重置索引一次
    df = df.reset_index()
    
    # 处理 Supersector 列
    df[' Benchmark ICB Supersector '] = df[' Benchmark ICB Supersector '].map(mapping_sector_to_id)
    
    # 当Supersector为0或NaN时，用FactSet Ind映射的值替换
    temp_icb19 = df['FactSet Ind'].map(mapping_factset_to_icb19)
    mask_sector_zero = (df[' Benchmark ICB Supersector '] == 0) | (df[' Benchmark ICB Supersector '].isna())
    df.loc[mask_sector_zero, ' Benchmark ICB Supersector '] = temp_icb19[mask_sector_zero]
    
    # 处理 Industry 列
    df[' Benchmark ICB Industry '] = df[' Benchmark ICB Industry '].map(mapping_industry_to_id)
    
    # 当Industry为0或NaN时，用Supersector映射到ICB11的值替换
    temp_icb11 = df[' Benchmark ICB Supersector '].map(mapping_icb19_to_icb11)
    mask_industry_zero = (df[' Benchmark ICB Industry '] == 0) | (df[' Benchmark ICB Industry '].isna())
    df.loc[mask_industry_zero, ' Benchmark ICB Industry '] = temp_icb11[mask_industry_zero]
    
    # 设置索引
    df.set_index('ISIN', inplace=True)
    
    # 日期转换
    df['Date'] = pd.to_datetime(df['Date'])
    
    return df

In [24]:
def add_icb_supersector_names(dataframe, icb_code_column=' Benchmark ICB Supersector '):
    """
    Add ICB Supersector names to a dataframe based on ICB code numbers.
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The dataframe containing ICB supersector codes
    icb_code_column : str, default=' Benchmark ICB Supersector '
        The name of the column containing ICB supersector codes
        
    Returns:
    --------
    pandas.DataFrame
        The dataframe with a new 'Supersector' column containing the ICB supersector names
    """
    # ICB Supersector mapping (name to code)
    icb_supersectors = {  
        "Auto & Parts": 1,  
        "Banks": 2,  
        "Basic Resources": 3,  
        "Chemicals": 4,  
        "Construction": 5,  
        "Financial Services": 6,  
        "Food, Beverage & Tobacco": 7,  
        "Health Care": 8,  
        "Industrial Goods & Services": 9,  
        "Insurance": 10,  
        "Media": 11,  
        "Energy": 12,  
        "Personal & Household Goods": 13,  
        "Real Estate": 14,  
        "Retail": 15,  
        "Technology": 16,  
        "Telecommunications": 17,  
        "Travel & Leisure": 18,  
        "Utilities": 19  
    }

    # Create a reverse mapping dictionary (code -> name)  
    icb_supersectors_reverse = {v: k for k, v in icb_supersectors.items()}  

    # Add a new column with the supersector name  
    dataframe_updated = dataframe.copy()
    dataframe_updated['Supersector'] = dataframe_updated[icb_code_column].map(icb_supersectors_reverse)
    
    return dataframe_updated

In [21]:
import pandas as pd
from presentation_layer import PresentationDataRepository
df = PresentationDataRepository().screen(last_only=True).copy()

df = factset_icb_mapping(df)


In [22]:
df.rename(columns={'Dividend Avg Percentile': 'Dividend Score (Histo + FY1)',
        'Value Avg Percentile' : 'Value Score (Histo + FY1)',
        'Quality Avg Percentile' : 'Quality Score (Histo + FY1)',
        'Growth Avg Percentile' : 'Growth Score (Histo + FY1)',
        'Mom Avg Percentile' : 'Momentum Score (Histo + FY1)',
        'Size Avg Percentile' : 'Size Score (Histo + FY1)',
        'LowVol Avg Percentile' : 'LowVol Score (Histo + FY1)',
        'Dividend_NTM Avg Percentile' : 'Dividend Score (Histo + NTM)',
        'Value_NTM Avg Percentile' : 'Value Score (Histo + NTM)',
        'Quality_NTM Avg Percentile' : 'Quality Score (Histo + NTM)',
        'Growth_NTM Avg Percentile' : 'Growth Score (Histo + NTM)',
        'Value_Forward Avg Percentile' : 'Value Score (FY1)',
        'Value_Spot_Avg Percentile' : 'Value Score (Histo)',
        'Value_NTM Avg Percentile.1' : 'Value Score (NTM)',
        'Growth_Forward_Avg Percentile' : 'Growth Score (FY1)',
        'Growth_Historical_Avg Percentile' : 'Growth Score (Histo)',
        'Growth_NTM_Avg Percentile': 'Growth Score (NTM)'}, inplace=True)

In [ ]:
# cols = ['Dividend Score (Histo + FY1)',
#         'Value Score (Histo + FY1)',
#         'Quality Score (Histo + FY1)',
#         'Growth Score (Histo + FY1)',
#         'Momentum Score (Histo + FY1)',
#         'Size Score (Histo + FY1)',
#         'LowVol Score (Histo + FY1)',
#         'Dividend Score (Histo + NTM)',
#         'Value Score (Histo + NTM)',
#         'Quality Score (Histo + NTM)',
#         'Growth Score (Histo + NTM)',
#         'Value Score (FY1)',
#         'Value Score (Histo)',
#         'Value Score (NTM)',
#         'Growth Score (FY1)',
#         'Growth Score (Histo)',
#         'Growth Score (NTM)']

if 'Supersecotr' not in df.columns:
        df = add_icb_supersector_names(df, icb_code_column=' Benchmark ICB Supersector ')

# df[cols] = df.groupby(["Exchange Country Region", 'Supersector'])[cols].rank(pct=True) * 10

In [26]:
df.drop(columns=[ 'FactSet Ind',
                    'FactSet Economy',
                    ' Benchmark ICB Industry ',
                    ' Benchmark ICB Supersector '], inplace=True)

In [27]:
df.reset_index(inplace=True)

In [28]:
import pandas as pd
import numpy as np

# ==========================================
# 0. 参数设置
# ==========================================
pe_col = 'PE NTM'
growth_col = 'EPS Forward Growth CGR3'
div_col = 'DVD Yield FY1'  # 新加入的列

# 分组键
group_cols = ['Exchange Country Region', 'Supersector']

# ==========================================
# 1. 数据预处理 (Pre-processing)
# ==========================================

# A. 处理 Growth (通用)
# ------------------------------------------
# 假设原始数据是小数 (0.15)，转为百分数 (15.0)
# 填充 NaN 为 0 (对 PEGY 很重要，对 PEG 稍后过滤)
raw_growth = df[growth_col].fillna(0) * 100
# 封顶 30% (防止画大饼)
raw_growth_capped = raw_growth.clip(upper=30.0)

# B. 处理 Yield (新增)
# ------------------------------------------
# 假设 Yield 也是小数 (0.04)，转为百分数 (4.0)
# 没有分红的填 0
raw_yield = df[div_col].fillna(0)

# ==========================================
# 2. 计算 PEG Ratio (你是怎么增长的？)
# ==========================================

# 这里的逻辑保持你原有的严格过滤：
# 如果 Growth <= 0.01，PEG 无意义 (设为 NaN)
peg_denominator = np.where(raw_growth_capped <= 0.01, np.nan, raw_growth_capped)

df['PEG_Raw'] = np.where(
    (df[pe_col] > 0) & (~np.isnan(peg_denominator)),
    df[pe_col] / peg_denominator,
    np.nan
)

# 清洗 PEG 极端值
df['PEG_Raw'] = df['PEG_Raw'].clip(lower=0.1, upper=5.0)

# ==========================================
# 3. 计算 PEGY Ratio (你是怎么回报股东的？)
# ==========================================

# 核心公式: PE / (Growth + Yield)
# 这里的逻辑更宽容：Growth 可以是 0，只要 Total Return > 1.0 即可
total_return = peg_denominator + raw_yield

df['PEGY_Raw'] = np.where(
    (df[pe_col] > 0) & (total_return > 1.0), # 分母至少要有 1% 的总回报
    df[pe_col] / total_return,
    np.nan
)

# 清洗 PEGY 极端值 (规则同 PEG)
df['PEGY_Raw'] = df['PEGY_Raw'].clip(lower=0.1, upper=5.0)

# ==========================================
# 4. Groupby Rank Scoring (0-10)
# ==========================================

# 定义一个打分函数，方便复用
def calculate_score(df, col_name, output_col):
    # 1. 计算百分位 (0.0 - 1.0)
    # 注意：这里我们只对非 NaN 的值排名
    pct = df.groupby(group_cols)[col_name].rank(pct=True, ascending=True)
    
    # 2. 转换为分数并反转 (10 = 最便宜/最好, 0 = 最贵)
    score = (1 - pct) * 10
    
    return score

# 计算双因子得分
df['PEG Percentile'] = calculate_score(df, 'PEG_Raw', 'Score_PEG')
df['PEGY Percentile'] = calculate_score(df, 'PEGY_Raw', 'Score_PEGY')

# ==========================================
# 5. (可选) 智能合成最终 Valuation Score
# ==========================================
# 逻辑：如果有 PEGY 分数（通常覆盖更广），优先参考 PEGY
# 或者取两者的均值
# df['PEG/PEGY Percentile'] = df[['PEG Percentile', 'PEGY Percentile']].mean(axis=1)

In [29]:
df_EU_US = df[df['Exchange Country Region'].isin(['West Europe', 'North America'])]
df_EU_US = df_EU_US[df_EU_US['Exchange Country Name'] != "Canada"]

In [30]:
df_EU_US.groupby(['Exchange Country Region', 'Supersector'])[["PE NTM", "PEG_Raw", "PEGY_Raw"]].median()

PE NTM   PEG_Raw  \
Exchange Country Region Supersector                                        
North America           Auto & Parts                 11.802570  0.341149   
                        Banks                         9.791583  2.618585   
                        Basic Resources              13.911575  4.783407   
                        Chemicals                    15.833460  0.812467   
                        Construction                 21.004140  0.917091   
                        Energy                       13.655275  0.791900   
                        Financial Services           13.287315  0.516800   
                        Food, Beverage & Tobacco     15.620090  5.000000   
                        Health Care                  18.568750  2.483802   
                        Industrial Goods & Services  18.742670  3.387715   
                        Insurance                    10.455970  0.902559   
                        Media                        16.496820  1.838971   
                        Personal & Household Goods   13.853650  0.385257   
                        Real Estate                  25.750270  1.633866   
                        Retail                       15.946210  1.007238   
                        Technology                   21.166520  1.182504   
                        Telecommunications           15.422230  1.527107   
                        Travel & Leisure             16.533600  1.122448   
                        Utilities                    18.567170  0.596796   
West Europe             Auto & Parts                  9.706825  0.472962   
                        Banks                        10.302695  1.800556   
                        Basic Resources              12.287270  1.810298   
                        Chemicals                    15.235975  3.433765   
                        Construction                 13.696790  2.227687   
                        Energy                        9.487323  1.234859   
                        Financial Services           12.005015  0.612242   
                        Food, Beverage & Tobacco     13.806630  2.128543   
                        Health Care                  18.104400  5.000000   
                        Industrial Goods & Services  16.830070  5.000000   
                        Insurance                    10.492280  0.586792   
                        Media                        11.355445  2.626009   
                        Personal & Household Goods   15.052205  3.068544   
                        Real Estate                  14.010850  0.993313   
                        Retail                       12.130010  0.907013   
                        Technology                   19.600415  2.955965   
                        Telecommunications           14.784680  0.835238   
                        Travel & Leisure             10.956150  0.843790   
                        Utilities                    13.913365  1.919467   

                                                     PEGY_Raw  
Exchange Country Region Supersector                            
North America           Auto & Parts                 0.341149  
                        Banks                        1.052928  
                        Basic Resources              1.181314  
                        Chemicals                    0.812467  
                        Construction                 0.908756  
                        Energy                       0.705096  
                        Financial Services           0.404297  
                        Food, Beverage & Tobacco     1.578233  
                        Health Care                  2.132128  
                        Industrial Goods & Services  1.255160  
                        Insurance                    0.838836  
                        Media                        0.838046  
                        Personal & Household Goods   0.385257  
                        Real Estate                  1.

In [31]:
df_EU_US.groupby(['Exchange Country Region', 'Supersector'])[["PE NTM", "PEG_Raw", "PEGY_Raw"]].median().to_excel(r"C:\GoogleDrive\TP\00_screen\PEG_PE_median_by_Supersector.xlsx")

In [32]:
pd.DataFrame(df[df['Name'].str.contains('UnitedHealth Group', case=False, na=False)].iloc[0, :]).dropna(axis=0).to_dict()

{10349: {'ISIN': 'US91324P1021',
  'Date': Timestamp('2025-11-30 00:00:00'),
  '5Y_Hist EPS TrendStab': 0.6279559,
  '5Y_Hist EPS TrendStab R2': 2.062524,
  '5Y_Hist EPS TrendStab slope': 0.30446,
  '5Y_Hist GrossInc TrendStab R2': 0.08920655,
  '5Y_Hist GrossInc TrendStab slope': 777.5068,
  '5Y_Hist Sales TrendStab': 38702.25,
  '5Y_Hist Sales TrendStab R2': 0.9760075,
  '5Y_Hist Sales TrendStab slope': 39653.64,
  'Asset TO exFIN': 1.37904,
  'Benchmark Country English': 'United States',
  'Benchmark Market Value Millions in EUR': 257393.8,
  'Benchmark Market Value Millions in EUR BK': 258056.1,
  'Company Main Exchange': 'S&P 500 Index',
  'Company SEDOL': 'GGPVTB-R',
  'Custom EV last': 335697.0,
  'DPS 1Y Growth FY1': 9.910941,
  'DPS 1Y Growth NTM': 6.52808,
  'DPS 5Y R2': 98.00848,
  'DPS 5Y Slope': 0.7880625,
  'DPS FY1': 8.588867,
  'DPS NTM': 9.074964,
  'DPS TrendStab': 77.23681,
  'DVD Payout FY0': 44.98416,
  'DVD Yield FY0': 2.747854,
  'DVD Yield FY1': 2.604502,
  'DVD